# Run performance experiments on BKT

## Imports

In [ ]:
!git clone https://github.com/CAHLR/pyBKT.git
%cd pyBKT
!pip install . --force-reinstall

Cloning into 'pyBKT'...
remote: Enumerating objects: 4721, done.
remote: Counting objects: 100% (261/261), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 4721 (delta 221), reused 207 (delta 194), pack-reused 4460 (from 3)
Receiving objects: 100% (4721/4721), 3.34 MiB | 9.96 MiB/s, done.
Resolving deltas: 100% (2251/2251), done.
/content/pyBKT
Processing /content/pyBKT
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.4 MB/s eta 0:00:00
Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from pyBKT.models import Model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/education-ml-research"

DATA_PATH_09 = os.path.join(
    BASE_DIR, "ASSISTments2009", "skill_builder_data.csv"
)

GROUPS_PATH_09 = os.path.join(
    BASE_DIR, "ASSISTments2009", "student_irt_ability_groups.csv"
)

print("Setup complete.")
print("Dataset exists:", os.path.exists(DATA_PATH_09))
print("IRT groups exist:", os.path.exists(GROUPS_PATH_09))

Mounted at /content/drive
Setup complete.
Dataset exists: True
IRT groups exist: True


## Load and clean ASSISTments 2009

In [ ]:
# Load ASSISTments 2009
assist09_raw = pd.read_csv(DATA_PATH_09, encoding="latin-1")

# Remove rows missing required fields
assist09 = assist09_raw.dropna(
    subset=["user_id", "skill_id", "correct", "order_id"]
).copy()

# Convert required columns to numeric
assist09["correct"] = pd.to_numeric(
    assist09["correct"], errors="coerce"
)
assist09["order_id"] = pd.to_numeric(
    assist09["order_id"], errors="coerce"
)
assist09["skill_id"] = pd.to_numeric(
    assist09["skill_id"], errors="coerce"
)

# Keep valid binary outcomes and required numeric values
assist09 = assist09.dropna(
    subset=["correct", "order_id", "skill_id"]
)
assist09 = assist09[assist09["correct"].isin([0, 1])].copy()

# Match the original experiment's skill format
assist09["skill_id"] = assist09["skill_id"].astype(int).astype(str)
assist09["user_id"] = assist09["user_id"].astype(str)

print("Raw rows:", len(assist09_raw))
print("Clean rows:", len(assist09))
print("Unique students:", assist09["user_id"].nunique())
print("Unique skills:", assist09["skill_id"].nunique())
print("Correct-response rate:", round(assist09["correct"].mean(), 4))

/tmp/ipykernel_3571/130243540.py:2: DtypeWarning: Columns (17: skill_name) have mixed types. Specify dtype option on import or set low_memory=False.
  assist09_raw = pd.read_csv(DATA_PATH_09, encoding="latin-1")


Raw rows: 525534
Clean rows: 459208
Unique students: 4163
Unique skills: 123
Correct-response rate: 0.6904


## Load IRT groups

In [ ]:
irt_groups_09 = pd.read_csv(GROUPS_PATH_09)

irt_groups_09["user_id"] = irt_groups_09["user_id"].astype(str)

print("IRT group rows:", len(irt_groups_09))
print("Unique students:", irt_groups_09["user_id"].nunique())
print("Columns:", irt_groups_09.columns.tolist())
print("\nQuartile counts:")
print(irt_groups_09["theta_quartile"].value_counts().sort_index())

print("\nMissing values:")
print(
    irt_groups_09[
        ["user_id", "theta", "theta_quartile", "theta_sd_group"]
    ].isna().sum()
)

IRT group rows: 4163
Unique students: 4163
Columns: ['user_id', 'theta', 'theta_quartile', 'theta_sd_group']

Quartile counts:
theta_quartile
Q1    1041
Q2    1041
Q3    1040
Q4    1041
Name: count, dtype: int64

Missing values:
user_id           0
theta             0
theta_quartile    0
theta_sd_group    0
dtype: int64


## Create 80/20 split

In [ ]:
train_students_09 = []
test_students_09 = []

for quartile in ["Q1", "Q2", "Q3", "Q4"]:
    students = (
        irt_groups_09.loc[
            irt_groups_09["theta_quartile"] == quartile,
            "user_id"
        ].to_numpy()
    )

    train_ids, test_ids = train_test_split(
        students,
        test_size=0.20,
        random_state=SEED
    )

    train_students_09.extend(train_ids)
    test_students_09.extend(test_ids)

print("Training students:", len(train_students_09))
print("Test students:", len(test_students_09))
print("Student overlap:", len(set(train_students_09) & set(test_students_09)))

print("\nTest students per quartile:")
print(
    irt_groups_09[
        irt_groups_09["user_id"].isin(test_students_09)
    ]["theta_quartile"].value_counts().sort_index()
)

Training students: 3328
Test students: 835
Student overlap: 0

Test students per quartile:
theta_quartile
Q1    209
Q2    209
Q3    208
Q4    209
Name: count, dtype: int64


## Verify the split

In [ ]:
train_set_09 = set(train_students_09)
test_set_09 = set(test_students_09)

assert train_set_09.isdisjoint(test_set_09)

for group_col in ["theta_quartile", "theta_sd_group"]:
    test_groups = irt_groups_09[
        irt_groups_09["user_id"].isin(test_set_09)
    ]

    print(f"\nTest students by {group_col}:")
    print(test_groups[group_col].value_counts().sort_index())

    train_groups = irt_groups_09[
        irt_groups_09["user_id"].isin(train_set_09)
    ]

    assert set(train_groups["user_id"]).isdisjoint(
        set(test_groups["user_id"])
    )

print("\nVerified: no training students appear in any test group.")


Test students by theta_quartile:
theta_quartile
Q1    209
Q2    209
Q3    208
Q4    209
Name: count, dtype: int64

Test students by theta_sd_group:
theta_sd_group
+0.5 to +1.0 SD     74
+1.0 to +1.5 SD     24
-0.5 to 0 SD       242
-1.0 to -0.5 SD    136
-1.5 to -1.0 SD     53
0 to +0.5 SD       183
< -1.5 SD           74
> +1.5 SD           49
Name: count, dtype: int64

Verified: no training students appear in any test group.


## Prepare data for BKT

In [ ]:
def prepare_bkt_data(data):
    data = data[
        ["order_id", "user_id", "skill_id", "correct"]
    ].copy()

    data = data.dropna(subset=["skill_id"])

    data["skill_id"] = data["skill_id"].astype(int).astype(str)

    data["correct"] = pd.to_numeric(data["correct"], errors="coerce")
    data["order_id"] = pd.to_numeric(data["order_id"], errors="coerce")

    data = data.dropna(subset=["correct", "order_id"])
    data = data[data["correct"].isin([0, 1])].copy()

    data = data.sort_values(
        ["user_id", "order_id"]
    ).reset_index(drop=True)

    return data


bkt_train_09 = prepare_bkt_data(
    assist09[assist09["user_id"].isin(train_set_09)]
)

bkt_test_09 = prepare_bkt_data(
    assist09[assist09["user_id"].isin(test_set_09)]
)

print("Training interactions:", len(bkt_train_09))
print("Test interactions:", len(bkt_test_09))
print("Training students:", bkt_train_09["user_id"].nunique())
print("Test students:", bkt_test_09["user_id"].nunique())
print(
    "Student overlap:",
    len(
        set(bkt_train_09["user_id"])
        & set(bkt_test_09["user_id"])
    )
)

Training interactions: 369313
Test interactions: 89895
Training students: 3328
Test students: 835
Student overlap: 0


## Fit BKT

In [ ]:
start = time.time()

bkt_model_09 = Model(
    seed=42,
    num_fits=1,
    parallel=True
)

bkt_model_09.fit(
    data=bkt_train_09,
    defaults={
        "order_id": "order_id",
        "student_id": "user_id",
        "skill_name": "skill_id",
        "correct": "correct"
    }
)

print(f"BKT training time: {time.time() - start:.2f} seconds")
print("BKT fitting complete.")

BKT training time: 1.74 seconds
BKT fitting complete.


## Evaluate BKT on the held-out test students

In [ ]:
# Predict on the held-out test set
bkt_test_predictions_09 = bkt_model_09.predict(data=bkt_test_09.copy())

print("Prediction rows:", len(bkt_test_predictions_09))
print("Test rows:", len(bkt_test_09))
print("Prediction columns:", bkt_test_predictions_09.columns.tolist())
print(
    "Missing predicted probabilities:",
    bkt_test_predictions_09["correct_predictions"].isna().sum()
)

Prediction rows: 89895
Test rows: 89895
Prediction columns: ['order_id', 'user_id', 'skill_id', 'correct', 'correct_predictions', 'state_predictions']
Missing predicted probabilities: 0


## Calculate AUC and Brier Score

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np

y_true_09 = bkt_test_predictions_09["correct"].astype(int)
y_prob_09 = bkt_test_predictions_09["correct_predictions"].astype(float)

print("Predicted probabilities finite:", np.isfinite(y_prob_09).all())
print("Predicted probabilities in [0, 1]:", y_prob_09.between(0, 1).all())
print("Overall test AUC:", round(roc_auc_score(y_true_09, y_prob_09), 6))
print("Overall test Brier Score:", round(brier_score_loss(y_true_09, y_prob_09), 6))

Predicted probabilities finite: True
Predicted probabilities in [0, 1]: True
Overall test AUC: 0.789549
Overall test Brier Score: 0.160652


In [ ]:
# Attach IRT ability groups to the test predictions
bkt_results_09 = bkt_test_predictions_09.merge(
    irt_groups_09[["user_id", "theta_quartile", "theta_sd_group"]],
    on="user_id",
    how="left",
    validate="many_to_one"
)

print("Rows without an ability group:", bkt_results_09["theta_quartile"].isna().sum())

quartile_results_09 = []

for group in ["Q1", "Q2", "Q3", "Q4"]:
    subset = bkt_results_09[bkt_results_09["theta_quartile"] == group]
    y_true = subset["correct"].astype(int)
    y_prob = subset["correct_predictions"].astype(float)

    quartile_results_09.append({
        "Ability Quartile": group,
        "Students": subset["user_id"].nunique(),
        "Interactions": len(subset),
        "AUC": roc_auc_score(y_true, y_prob) if y_true.nunique() == 2 else np.nan,
        "Brier Score": brier_score_loss(y_true, y_prob)
    })

quartile_results_09 = pd.DataFrame(quartile_results_09)
print(quartile_results_09.round(6).to_string(index=False))

Rows without an ability group: 0
Ability Quartile  Students  Interactions     AUC  Brier Score
              Q1       209         10085 0.70701      0.22684
              Q2       209         21332 0.79992      0.17484
              Q3       208         39941 0.74151      0.15508
              Q4       209         18537 0.71943      0.12032


In [ ]:
sd_order_09 = [
    "< -1.5 SD",
    "-1.5 to -1.0 SD",
    "-1.0 to -0.5 SD",
    "-0.5 to 0 SD",
    "0 to +0.5 SD",
    "+0.5 to +1.0 SD",
    "+1.0 to +1.5 SD",
    "> +1.5 SD"
]

sd_results_09 = []

for group in sd_order_09:
    subset = bkt_results_09[bkt_results_09["theta_sd_group"] == group]
    if subset.empty:
        continue

    y_true = subset["correct"].astype(int)
    y_prob = subset["correct_predictions"].astype(float)

    sd_results_09.append({
        "Ability Group": group,
        "Students": subset["user_id"].nunique(),
        "Interactions": len(subset),
        "AUC": roc_auc_score(y_true, y_prob) if y_true.nunique() == 2 else np.nan,
        "Brier Score": brier_score_loss(y_true, y_prob)
    })

sd_results_09 = pd.DataFrame(sd_results_09)
print(sd_results_09.round(6).to_string(index=False))

  Ability Group  Students  Interactions     AUC  Brier Score
      < -1.5 SD        74          1758 0.67902      0.18498
-1.5 to -1.0 SD        53          2349 0.70381      0.22212
-1.0 to -0.5 SD       136          8919 0.66585      0.24230
   -0.5 to 0 SD       242         31112 0.78891      0.16857
   0 to +0.5 SD       183         37413 0.72615      0.14700
+0.5 to +1.0 SD        74          5133 0.68277      0.10452
+1.0 to +1.5 SD        24          2384 0.79955      0.05005
      > +1.5 SD        49           827 0.97458      0.04061


## BKT Results — ASSISTments 2009

### Overall Performance

| Metric | Score |
|---|---:|
| AUC | 0.789549 |
| Brier Score | 0.160652 |

### Performance by IRT Ability Quartile

| Ability Group | Students | Interactions | AUC | Brier Score |
|---|---:|---:|---:|---:|
| Q1 (Lowest θ) | 209 | 10,085 | 0.707010 | 0.226840 |
| Q2 | 209 | 21,332 | 0.799920 | 0.174840 |
| Q3 | 208 | 39,941 | 0.741510 | 0.155080 |
| Q4 (Highest θ) | 209 | 18,537 | 0.719430 | 0.120320 |

### Performance by IRT Standard-Deviation Group

| Ability Group | Students | Interactions | AUC | Brier Score |
|---|---:|---:|---:|---:|
| < −1.5 SD | 74 | 1,758 | 0.679020 | 0.184980 |
| −1.5 to −1.0 SD | 53 | 2,349 | 0.703810 | 0.222120 |
| −1.0 to −0.5 SD | 136 | 8,919 | 0.665850 | 0.242300 |
| −0.5 to 0 SD | 242 | 31,112 | 0.788910 | 0.168570 |
| 0 to +0.5 SD | 183 | 37,413 | 0.726150 | 0.147000 |
| +0.5 to +1.0 SD | 74 | 5,133 | 0.682770 | 0.104520 |
| +1.0 to +1.5 SD | 24 | 2,384 | 0.799550 | 0.050050 |
| > +1.5 SD | 49 | 827 | 0.974580 | 0.040610 |